# 🔋 RAG System for Energy Document Intelligence

**Subject:** Gen AI &nbsp;|&nbsp; **Tools:** Python · ChromaDB · Google Gemini API

---

## 📌 Project Overview

This notebook implements a complete **Retrieval-Augmented Generation (RAG)** pipeline for energy sector documents.
Users upload PDF reports and manuals, which are processed through:

1. **Text Extraction** — Parse raw text from PDFs page by page
2. **Chunking** — Split text into overlapping semantic chunks
3. **Vector Embedding** — Embed chunks using Google's `text-embedding-004` model
4. **Vector Storage** — Store embeddings in ChromaDB for fast similarity search
5. **Grounded Q&A** — Retrieve relevant context and answer via Gemini Pro

All responses are **strictly grounded** in the uploaded documents — the LLM cannot use outside knowledge.

---

## 🏗️ High Level Design (HLD)

```
┌─────────────────────────────────────────────────────────────────┐
│                        RAG PIPELINE                             │
│                                                                 │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌───────────┐  │
│  │  PDF     │──▶│  Text    │──▶│  Chunk   │──▶│  Embed &  │  │
│  │  Upload  │   │  Extract │   │  & Clean │   │  Store    │  │
│  └──────────┘   └──────────┘   └──────────┘   └─────┬─────┘  │
│                                                       │        │
│                  INGESTION PHASE                      │        │
│  ─────────────────────────────────────────────────── │ ──     │
│                  QUERY PHASE                          │        │
│                                                       ▼        │
│  ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌───────────┐  │
│  │  User    │──▶│  Embed   │──▶│ ChromaDB │──▶│  Gemini   │  │
│  │  Query   │   │  Query   │   │ Top-K    │   │  Pro LLM  │  │
│  └──────────┘   └──────────┘   └──────────┘   └─────┬─────┘  │
│                                                       │        │
│                                               ┌───────▼─────┐  │
│                                               │  Grounded   │  │
│                                               │  Answer +   │  │
│                                               │  Citations  │  │
│                                               └─────────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

---

## 🔬 Low Level Design (LLD)

| Module | Class / Function | Responsibility |
|---|---|---|
| **PDFProcessor** | `extract_text()` | PyMuPDF page-by-page extraction |
| **TextChunker** | `chunk()` | Sliding window with token overlap |
| **GeminiEmbedder** | `embed_texts()`, `embed_query()` | Batch embedding via Gemini API |
| **VectorStore** | `add_documents()`, `search()` | ChromaDB cosine similarity store |
| **RAGChain** | `ingest()`, `query()` | Orchestrates full pipeline |
| **EnergyRAGSystem** | `upload_pdf()`, `ask()` | Top-level user-facing interface |

**Data flow for a query:**
```
question (str)
    → GeminiEmbedder.embed_query()        → query_vector [768 floats]
    → VectorStore.search(query_vector)     → top_k chunks with scores
    → RAGChain._build_prompt()             → grounded prompt string
    → Gemini Pro generate_content()        → answer + citations
```

---
## ⚙️ Section 1 — Installation & Setup

In [1]:
# Install all required libraries
# Run this cell once — restart kernel after if prompted

%pip install -q \
    google-genai \
    pymupdf \
    chromadb \
    tiktoken \
    ipywidgets \
    tqdm \
    sentence-transformers

print("✅ All packages installed successfully")

Note: you may need to restart the kernel to use updated packages.
✅ All packages installed successfully



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ── Standard library ──────────────────────────────────────────────
import os
import re
import uuid
import json
import textwrap
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Dict

# ── Third-party ───────────────────────────────────────────────────
import fitz                          # PyMuPDF  — PDF text extraction
import tiktoken                      # Token counting for chunking
import chromadb                      # Local vector database
from chromadb.config import Settings as ChromaSettings
from google import genai             # Gemini LLM
from sentence_transformers import SentenceTransformer  # Local embeddings
from tqdm.notebook import tqdm       # Progress bars
from IPython.display import display, Markdown, HTML

print("✅ All imports successful")

✅ All imports successful


In [ ]:
# ─────────────────────────────────────────────────────────────────
#  🔑  CONFIGURE YOUR GOOGLE GEMINI API KEY
#
#  Get a free key at: https://aistudio.google.com/app/apikey
#  Paste it below between the quotes
# ─────────────────────────────────────────────────────────────────

GEMINI_API_KEY = "KEY_HERE"   # ← paste your key here

# ─── create the Gemini client (new google-genai SDK style) ───────
client = genai.Client(api_key=GEMINI_API_KEY, http_options={'api_version': 'v1'})

# ─── Global model settings ───────────────────────────────────────
# ── List available embedding models ─────────────────────────────
# Run this to see exactly which models your API key can access
for m in client.models.list():
    if 'embed' in m.name.lower():
        print(m.name)

# ─── Set this to whichever embedding model appears above ─────────
EMBEDDING_MODEL = "all-MiniLM-L6-v2"           # Local sentence-transformers model (no API needed)
LLM_MODEL       = "gemini-2.5-flash"           # Gemini model (from your available list)
CHROMA_PATH     = "./chroma_db"                # Where ChromaDB stores vectors
CHUNK_SIZE      = 500                          # Max tokens per chunk
CHUNK_OVERLAP   = 50                           # Overlap tokens between chunks
TOP_K           = 8                            # Chunks to retrieve per query

print(f"✅ Gemini configured")
print(f"   LLM          : {LLM_MODEL}")
print(f"   Embeddings   : {EMBEDDING_MODEL}")
print(f"   Vector store : {CHROMA_PATH}")
print(f"   Chunk size   : {CHUNK_SIZE} tokens (overlap: {CHUNK_OVERLAP})")

✅ Gemini configured
   LLM          : gemini-2.5-flash
   Embeddings   : all-MiniLM-L6-v2
   Vector store : ./chroma_db
   Chunk size   : 500 tokens (overlap: 50)


---
## 📄 Section 2 — PDF Processor
Extracts text from every page of a PDF using **PyMuPDF**. Returns structured page objects so we preserve page numbers for citations.

In [17]:
@dataclass
class Page:
    """Represents a single extracted PDF page."""
    page_no: int
    text: str
    doc_name: str


class PDFProcessor:
    """
    Extracts and cleans text from PDF documents page by page.
    Uses PyMuPDF (fitz) for high-fidelity text extraction.
    """

    def extract(self, pdf_path: str) -> List[Page]:
        """Extract text from all pages of a PDF.
        
        Args:
            pdf_path: Absolute or relative path to the PDF file.
        Returns:
            List of Page objects, one per non-empty page.
        """
        path = Path(pdf_path)
        if not path.exists():
            raise FileNotFoundError(f"PDF not found: {pdf_path}")
        if path.suffix.lower() != ".pdf":
            raise ValueError(f"Expected a .pdf file, got: {path.suffix}")

        doc_name = path.stem
        pages: List[Page] = []

        with fitz.open(pdf_path) as doc:
            for i, page in enumerate(doc):
                raw_text = page.get_text("text")
                cleaned  = self._clean(raw_text)
                if cleaned:   # skip blank pages
                    pages.append(Page(
                        page_no  = i + 1,
                        text     = cleaned,
                        doc_name = doc_name,
                    ))

        print(f"   📄 Extracted {len(pages)} pages from '{path.name}'")
        return pages

    def _clean(self, text: str) -> str:
        """Remove common PDF noise: broken hyphenation, excessive whitespace."""
        text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)   # fix line-break hyphenation
        text = re.sub(r"\n{3,}", "\n\n", text)           # collapse blank lines
        text = re.sub(r"[ \t]{3,}", " ", text)           # collapse wide spaces
        text = re.sub(r"(?i)page\s+\d+", "", text)       # strip "Page 12" artifacts
        return text.strip()


# ── Quick test ────────────────────────────────────────────────────
print("✅ PDFProcessor defined")
print("   Usage: processor = PDFProcessor()")
print("          pages = processor.extract('your_document.pdf')")

✅ PDFProcessor defined
   Usage: processor = PDFProcessor()
          pages = processor.extract('your_document.pdf')


---
## ✂️ Section 3 — Text Chunker
Splits page text into overlapping token-based chunks using a **sliding window**. Overlap prevents context from being cut off at chunk boundaries.

In [18]:
@dataclass
class Chunk:
    """A single text chunk ready for embedding."""
    chunk_id  : str
    doc_name  : str
    page_no   : int
    text      : str
    token_count: int


class TextChunker:
    """
    Sliding-window token-based chunker.

    Strategy
    --------
    1. Tokenize page text with tiktoken (cl100k_base — same as GPT/Gemini).
    2. Walk through tokens with a window of `chunk_size`.
    3. Advance by (chunk_size - overlap) each step.
    4. Decode each window back to text and create a Chunk object.
    """

    def __init__(self, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
        self.chunk_size = chunk_size
        self.overlap    = overlap
        self._enc       = tiktoken.get_encoding("cl100k_base")

    def chunk(self, pages: List[Page]) -> List[Chunk]:
        """Chunk all pages into overlapping token windows.
        
        Args:
            pages: Output of PDFProcessor.extract()
        Returns:
            List of Chunk objects ready for embedding.
        """
        all_chunks: List[Chunk] = []
        step = self.chunk_size - self.overlap

        for page in pages:
            tokens = self._enc.encode(page.text)

            # If page is shorter than chunk_size → single chunk
            if len(tokens) <= self.chunk_size:
                all_chunks.append(Chunk(
                    chunk_id    = str(uuid.uuid4()),
                    doc_name    = page.doc_name,
                    page_no     = page.page_no,
                    text        = page.text,
                    token_count = len(tokens),
                ))
                continue

            # Sliding window
            for start in range(0, len(tokens) - self.overlap, step):
                end        = min(start + self.chunk_size, len(tokens))
                window     = tokens[start:end]
                chunk_text = self._enc.decode(window).strip()
                if chunk_text:
                    all_chunks.append(Chunk(
                        chunk_id    = str(uuid.uuid4()),
                        doc_name    = page.doc_name,
                        page_no     = page.page_no,
                        text        = chunk_text,
                        token_count = len(window),
                    ))
                if end == len(tokens):
                    break

        return all_chunks


print("✅ TextChunker defined")
print(f"   Default window : {CHUNK_SIZE} tokens")
print(f"   Default overlap: {CHUNK_OVERLAP} tokens")

✅ TextChunker defined
   Default window : 500 tokens
   Default overlap: 50 tokens


---
## 🔢 Section 4 — Gemini Embedder
Converts text chunks into **768-dimensional dense vectors** using Google's `text-embedding-004` model. These vectors capture semantic meaning — similar texts cluster close together in vector space.

In [19]:
class GeminiEmbedder:
    """
    Generates text embeddings locally using sentence-transformers.

    Uses 'all-MiniLM-L6-v2' — a fast, lightweight model that produces
    384-dimensional embeddings. No API key or internet needed after
    the first download (~90MB cached locally).
    """

    def __init__(self, model: str = EMBEDDING_MODEL):
        print(f"   Loading embedding model '{model}' (downloads once, ~90MB)...")
        self.model_name = model
        self._model = SentenceTransformer(model)
        print(f"   ✅ Embedding model ready")

    def embed_texts(self, texts: List[str], show_progress: bool = True) -> List[List[float]]:
        """Embed a list of document chunks.

        Args:
            texts: List of chunk strings to embed.
            show_progress: Show tqdm progress bar.
        Returns:
            List of embedding vectors (each is a list of 384 floats).
        """
        embeddings = self._model.encode(
            texts,
            show_progress_bar    = show_progress,
            convert_to_numpy     = True,
            normalize_embeddings = True,
        )
        return embeddings.tolist()

    def embed_query(self, query: str) -> List[float]:
        """Embed a single user query for retrieval."""
        embedding = self._model.encode(
            query,
            convert_to_numpy     = True,
            normalize_embeddings = True,
        )
        return embedding.tolist()


print("✅ GeminiEmbedder defined (using local sentence-transformers)")
print(f"   Model      : {EMBEDDING_MODEL}")
print(f"   Output dims: 384 floats per text")
print(f"   API needed : None — runs fully locally")

✅ GeminiEmbedder defined (using local sentence-transformers)
   Model      : all-MiniLM-L6-v2
   Output dims: 384 floats per text
   API needed : None — runs fully locally


---
## 🗄️ Section 5 — Vector Store (ChromaDB)
Stores embeddings locally using **ChromaDB** with cosine similarity. Supports adding documents, searching by vector similarity, and listing what's stored.

In [20]:
@dataclass
class SearchResult:
    """A retrieved chunk with its relevance score."""
    chunk_id : str
    doc_name : str
    page_no  : int
    text     : str
    score    : float   # cosine similarity — higher = more relevant


class VectorStore:
    """
    ChromaDB-backed persistent vector store.

    Stores chunk embeddings on disk so you don't need to
    re-embed documents every time the notebook restarts.
    Collection uses cosine similarity (hnsw:space = cosine).
    """

    COLLECTION = "energy_docs"

    def __init__(self, persist_path: str = CHROMA_PATH):
        self._client = chromadb.PersistentClient(
            path     = persist_path,
            settings = ChromaSettings(anonymized_telemetry=False),
        )
        self._col = self._client.get_or_create_collection(
            name     = self.COLLECTION,
            metadata = {"hnsw:space": "cosine"},
        )
        print(f"   📦 ChromaDB ready — {self._col.count()} chunks already stored")

    # ── Write ────────────────────────────────────────────────────

    def add_chunks(self, chunks: List[Chunk], embeddings: List[List[float]]) -> None:
        """Store chunks and their embeddings in ChromaDB."""
        self._col.add(
            ids        = [c.chunk_id for c in chunks],
            embeddings = embeddings,
            documents  = [c.text for c in chunks],
            metadatas  = [
                {"doc_name": c.doc_name, "page_no": c.page_no, "token_count": c.token_count}
                for c in chunks
            ],
        )
        print(f"   ✅ Stored {len(chunks)} chunks — total in DB: {self._col.count()}")

    # ── Read ─────────────────────────────────────────────────────

    def search(self, query_embedding: List[float], top_k: int = TOP_K) -> List[SearchResult]:
        """Find the top_k most similar chunks using cosine similarity."""
        results = self._col.query(
            query_embeddings = [query_embedding],
            n_results        = min(top_k, self._col.count()),
            include          = ["documents", "metadatas", "distances"],
        )

        out: List[SearchResult] = []
        for i, cid in enumerate(results["ids"][0]):
            meta  = results["metadatas"][0][i]
            dist  = results["distances"][0][i]
            out.append(SearchResult(
                chunk_id = cid,
                doc_name = meta["doc_name"],
                page_no  = meta["page_no"],
                text     = results["documents"][0][i],
                score    = round(1 - dist, 4),   # convert distance → similarity
            ))
        return out

    def list_documents(self) -> List[str]:
        """Return unique document names currently indexed."""
        if self._col.count() == 0:
            return []
        all_meta = self._col.get(include=["metadatas"])["metadatas"]
        return sorted(set(m["doc_name"] for m in all_meta))

    def delete_document(self, doc_name: str) -> None:
        """Remove all chunks belonging to a document."""
        self._col.delete(where={"doc_name": doc_name})
        print(f"   🗑️  Deleted all chunks for '{doc_name}'")

    def count(self) -> int:
        return self._col.count()


print("✅ VectorStore defined")

✅ VectorStore defined


---
## 🔗 Section 6 — RAG Chain (Retrieval + Generation)
The core of the system. Retrieves relevant chunks and sends them to **Gemini Pro** with a strict system prompt that prevents hallucination.

In [21]:
# ── System prompt — the key to grounded, reliable answers ────────
SYSTEM_PROMPT = """You are an expert assistant for energy industry professionals.
You help analysts, engineers, and managers understand technical energy documents.

STRICT RULES YOU MUST FOLLOW:
1. Answer ONLY using the document excerpts provided below.
2. If the answer is NOT present in the excerpts, respond with:
   "I could not find this information in the uploaded documents."
3. Always cite your sources at the end in this format:
   📎 Sources: [Document Name, Page X] | [Document Name, Page Y]
4. Be precise, concise, and professional.
5. Never invent, extrapolate, or guess facts not in the context."""


class RAGChain:
    """
    Orchestrates the full Retrieval-Augmented Generation pipeline:
    PDF → Chunks → Embeddings → Vector DB → Retrieval → LLM → Answer
    """

    def __init__(self):
        self.processor = PDFProcessor()
        self.chunker   = TextChunker()
        self.embedder  = GeminiEmbedder()
        self.store     = VectorStore()

    # ── INGESTION ─────────────────────────────────────────────────

    def ingest(self, pdf_path: str) -> Dict:
        """Full ingestion pipeline for one PDF.
        
        Steps:
            1. Extract text from all pages (PyMuPDF)
            2. Split into overlapping chunks (TextChunker)
            3. Embed all chunks (Gemini text-embedding-004)
            4. Store vectors in ChromaDB
        
        Returns:
            dict with ingestion statistics
        """
        print(f"\n📥 Ingesting: {Path(pdf_path).name}")
        print("─" * 50)

        # Step 1 — Extract
        print("Step 1/4: Extracting text from PDF...")
        pages = self.processor.extract(pdf_path)

        # Step 2 — Chunk
        print("Step 2/4: Chunking text...")
        chunks = self.chunker.chunk(pages)
        print(f"   ✂️  Created {len(chunks)} chunks from {len(pages)} pages")

        # Step 3 — Embed
        print("Step 3/4: Generating embeddings (this may take a moment)...")
        texts      = [c.text for c in chunks]
        embeddings = self.embedder.embed_texts(texts)

        # Step 4 — Store
        print("Step 4/4: Storing in ChromaDB...")
        self.store.add_chunks(chunks, embeddings)

        stats = {
            "document" : Path(pdf_path).name,
            "pages"    : len(pages),
            "chunks"   : len(chunks),
            "total_db" : self.store.count(),
        }
        print(f"\n🎉 Ingestion complete! {stats}")
        return stats

    # ── QUERY ─────────────────────────────────────────────────────

    def query(self, question: str, top_k: int = TOP_K) -> Dict:
        """Answer a question using retrieved document context.
        
        Steps:
            1. Embed the question
            2. Retrieve top_k similar chunks from ChromaDB
            3. Build a grounded prompt with context
            4. Generate answer with Gemini Pro
        
        Returns:
            dict with 'answer', 'sources', and 'retrieved_chunks'
        """
        if self.store.count() == 0:
            return {"answer": "⚠️ No documents indexed yet. Please run ingest() first.",
                    "sources": [], "retrieved_chunks": []}

        # Step 1 — embed question
        query_vec = self.embedder.embed_query(question)

        # Step 2 — retrieve
        results = self.store.search(query_vec, top_k=top_k)

        # Step 3 — build prompt
        prompt = self._build_prompt(question, results)

        # Step 4 — generate
        response = client.models.generate_content(
            model    = LLM_MODEL,
            contents = f"{SYSTEM_PROMPT}\n\n{prompt}",
            config   = {"temperature": 0.0},
        )

        sources = [
            {"doc": r.doc_name, "page": r.page_no, "score": r.score}
            for r in results
        ]

        return {
            "answer"           : response.text,
            "sources"          : sources,
            "retrieved_chunks" : results,
        }

    def _build_prompt(self, question: str, chunks: List[SearchResult]) -> str:
        """Construct a context-rich prompt from retrieved chunks."""
        sections = []
        for i, c in enumerate(chunks, 1):
            sections.append(
                f"--- EXCERPT {i} ---\n"
                f"Document : {c.doc_name}  |  Page: {c.page_no}  |  Relevance: {c.score}\n"
                f"{c.text.strip()}"
            )
        context = "\n\n".join(sections)
        return (
            f"DOCUMENT EXCERPTS:\n\n{context}\n\n"
            f"{'='*60}\n\n"
            f"QUESTION: {question}\n\n"
            f"ANSWER (cite your sources):"
        )


print("✅ RAGChain defined")

✅ RAGChain defined


---
## 🎨 Section 7 — User Interface Helper
A clean display layer to format answers nicely in the notebook.

In [22]:
class EnergyRAGSystem:
    """
    Top-level interface for the Energy RAG System.
    Wraps RAGChain with formatted notebook output.
    """

    def __init__(self):
        print("🔋 Initialising Energy RAG System...")
        self._rag = RAGChain()
        print("✅ System ready!")
        self._show_indexed()

    def upload_pdf(self, path: str) -> None:
        """Ingest a PDF into the system."""
        self._rag.ingest(path)
        self._show_indexed()

    def ask(self, question: str, show_chunks: bool = False) -> None:
        """Ask a question and display the grounded answer."""
        display(HTML(f"""
        <div style='background:#f0f4ff;border-left:4px solid #4361ee;
                    padding:12px 16px;border-radius:4px;margin:8px 0'>
            <b>❓ Question:</b> {question}
        </div>"""))

        result = self._rag.query(question)

        display(HTML(f"""
        <div style='background:#f0fff4;border-left:4px solid #2dc653;
                    padding:12px 16px;border-radius:4px;margin:8px 0'>
            <b>💡 Answer:</b><br><br>
            {result['answer'].replace(chr(10), '<br>')}
        </div>"""))

        if result["sources"]:
            src_html = "".join(
                f"<span style='background:#e2e8f0;padding:2px 8px;border-radius:12px;"
                f"margin:2px;display:inline-block;font-size:13px'>"
                f"📄 {s['doc']} · p.{s['page']} · {s['score']:.2f}</span>"
                for s in result["sources"]
            )
            display(HTML(f"<div style='margin:4px 0'><b>📎 Retrieved from:</b><br>{src_html}</div>"))

        if show_chunks:
            print("\n── Retrieved Chunks ─────────────────────────────────")
            for i, chunk in enumerate(result["retrieved_chunks"], 1):
                print(f"\n[{i}] {chunk.doc_name} · Page {chunk.page_no} · Score {chunk.score}")
                print(textwrap.fill(chunk.text[:300] + "...", width=80))

    def list_documents(self) -> None:
        """Show all indexed documents."""
        self._show_indexed()

    def remove_document(self, doc_name: str) -> None:
        """Remove a document from the vector store by its name (without .pdf)."""
        self._rag.store.delete_document(doc_name)
        self._show_indexed()

    def _show_indexed(self):
        docs = self._rag.store.list_documents()
        if docs:
            display(HTML(
                "<b>📚 Indexed documents:</b> " +
                ", ".join(f"<code>{d}</code>" for d in docs) +
                f" &nbsp;|&nbsp; <b>{self._rag.store.count()} total chunks</b>"
            ))
        else:
            display(HTML("<i>📂 No documents indexed yet. Use <code>system.upload_pdf('file.pdf')</code></i>"))


print("✅ EnergyRAGSystem defined")

✅ EnergyRAGSystem defined


---
## 🚀 Section 8 — Run the System

Everything is now defined. Initialise the system and start ingesting documents!

In [23]:
# ── Initialise the system ─────────────────────────────────────────
# ChromaDB will load from disk if it already has data

system = EnergyRAGSystem()

🔋 Initialising Energy RAG System...
   Loading embedding model 'all-MiniLM-L6-v2' (downloads once, ~90MB)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   ✅ Embedding model ready
   📦 ChromaDB ready — 8 chunks already stored
✅ System ready!


In [24]:
# ── Upload a PDF document ─────────────────────────────────────────
#
#  Place your PDF in the same folder as this notebook, then run:
#
#    system.upload_pdf("your_energy_report.pdf")
#
#  You can call this multiple times to add more documents.
#  ChromaDB persists them — no need to re-upload after restarting.
# ─────────────────────────────────────────────────────────────────

system.upload_pdf("C:\\Users\\Srijan\\Downloads\\NorthSea_Energy_Q3_Report.pdf")   # ← replace with your filename


📥 Ingesting: NorthSea_Energy_Q3_Report.pdf
──────────────────────────────────────────────────
Step 1/4: Extracting text from PDF...
   📄 Extracted 7 pages from 'NorthSea_Energy_Q3_Report.pdf'
Step 2/4: Chunking text...
   ✂️  Created 8 chunks from 7 pages
Step 3/4: Generating embeddings (this may take a moment)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Step 4/4: Storing in ChromaDB...
   ✅ Stored 8 chunks — total in DB: 16

🎉 Ingestion complete! {'document': 'NorthSea_Energy_Q3_Report.pdf', 'pages': 7, 'chunks': 8, 'total_db': 16}


In [25]:
# ── Demo: Upload multiple documents ──────────────────────────────
#  Uncomment and edit paths as needed

# system.upload_pdf("Q3_Operations_Report.pdf")
# system.upload_pdf("Gas_Turbine_Manual.pdf")
# system.upload_pdf("Regulatory_Filing_2024.pdf")

---
## ❓ Section 9 — Ask Questions

Now query your documents with natural language. The system retrieves the most relevant excerpts and generates a grounded answer.

In [26]:
# ── Ask a question ────────────────────────────────────────────────
system.ask("What is the total electricity generation capacity mentioned in the document?")

In [27]:
system.ask("What are the key safety protocols described for equipment maintenance?")

In [14]:
system.ask("Summarise the main findings of this document.")

In [15]:
# ── Ask with chunk inspection (shows raw retrieved context) ──────
system.ask(
    "What environmental regulations are mentioned?",
    show_chunks=True   # ← set True to inspect retrieved context
)


── Retrieved Chunks ─────────────────────────────────

[1] NorthSea_Energy_Q3_Report · Page 6 · Score 0.409
5. Regulatory Compliance NSEC operations are regulated under the UK North Sea
Transition Authority (NSTA) and the Norwegian Petroleum Directorate (NPD)
frameworks. All regulatory reporting obligations for Q3 2024 were met on time
and without exception. Key compliance activities in the quarter: • Ann...

[2] NorthSea_Energy_Q3_Report · Page 4 · Score 0.388
3. Health, Safety & Environment (HSE) NSEC is committed to achieving zero harm
across all operations. Q3 2024 marked the ninth consecutive quarter without a
Lost Time Incident (LTI) across the entire operated portfolio — a significant
achievement representing over 4.2 million man-hours worked safely...

[3] NorthSea_Energy_Q3_Report · Page 1 · Score 0.2867
NORTH SEA ENERGY CONSORTIUM Q3 2024 Operations & Production Report Confidential
| For Internal Distribution Only Report Period: July – September 2024 Report
Date: October 15,

In [28]:
# ── Ask a question ────────────────────────────────────────────────
system.ask("What is the report period for this report")

---
## 🔍 Section 10 — Internals Inspection
Useful for assignments: show the pipeline internals step by step.

In [ ]:
# ── Inspect chunking output ───────────────────────────────────────
# This cell shows exactly what the chunker produces

# Example with synthetic text (replace with real PDF pages)
sample_pages = [
    Page(page_no=1, doc_name="demo",
         text="Natural gas production in the North Sea reached 45 billion cubic metres in 2023. "
              "This represents a 12% increase from the previous year driven by new offshore platforms. "
              "The main operators reported improved extraction efficiency through advanced drilling techniques. "
              "Safety incidents decreased by 30% due to updated operational protocols and staff training. "
              "Environmental compliance scores improved across all monitored sites." * 3)
]

chunker = TextChunker(chunk_size=100, overlap=20)
chunks  = chunker.chunk(sample_pages)

print(f"Input text tokens  : ~{len(sample_pages[0].text.split()) * 1} words")
print(f"Chunks produced    : {len(chunks)}")
print(f"Chunk size setting : 100 tokens  |  Overlap: 20 tokens")
print()
for i, c in enumerate(chunks[:3], 1):
    print(f"── Chunk {i} ({c.token_count} tokens) ──")
    print(textwrap.fill(c.text[:200], width=80))
    print()

In [ ]:
# ── Inspect an embedding vector ───────────────────────────────────
# Shows what a Gemini embedding looks like

embedder  = GeminiEmbedder()
test_vec  = embedder.embed_query("What is the gas production capacity?")

print(f"Query embedding shape : {len(test_vec)} dimensions")
print(f"First 10 values       : {[round(v, 5) for v in test_vec[:10]]}")
print(f"Min value             : {min(test_vec):.5f}")
print(f"Max value             : {max(test_vec):.5f}")
print(f"\nInterpretation: Each of the 768 floats captures a different")
print(f"semantic dimension. Similar texts will have vectors that point")
print(f"in the same direction (high cosine similarity score ≈ 1.0)")

In [ ]:
# ── Inspect the vector database ───────────────────────────────────

store = system._rag.store
print(f"Total chunks stored  : {store.count()}")
print(f"Indexed documents    : {store.list_documents()}")

if store.count() > 0:
    sample = store._col.get(limit=2, include=["documents", "metadatas"])
    print(f"\nSample stored chunk:")
    print(f"  Metadata : {sample['metadatas'][0]}")
    print(f"  Text     : {sample['documents'][0][:150]}...")

---
## 🗑️ Section 11 — Document Management

In [ ]:
# ── List all indexed documents ────────────────────────────────────
system.list_documents()

# ── Remove a document (use the filename without .pdf) ────────────
# system.remove_document("your_energy_report")

# ── Nuclear option: wipe the entire database ──────────────────────
# import shutil
# shutil.rmtree("./chroma_db", ignore_errors=True)
# print("🧹 Vector database cleared")

---
## 📋 Summary

| Component | Technology | Role |
|---|---|---|
| PDF Extraction | PyMuPDF (`fitz`) | Parse raw text from PDF pages |
| Text Chunking | tiktoken + sliding window | Split text into 500-token overlapping chunks |
| Embeddings | Google `text-embedding-004` | Convert text → 768-dim semantic vectors |
| Vector Store | ChromaDB (cosine similarity) | Store and search vectors on disk |
| LLM | Google `gemini-2.0-flash` | Generate grounded answers from context |
| Grounding | System prompt strict rules | Prevents hallucination beyond uploaded docs |

### Key Design Decisions
- **Overlapping chunks** prevent context from being severed at chunk boundaries
- **`task_type` differentiation** (RETRIEVAL_DOCUMENT vs RETRIEVAL_QUERY) improves search accuracy
- **Temperature = 0** on the LLM forces deterministic, factual responses
- **Persistent ChromaDB** means you only embed once — restarts load from disk
- **Page-level metadata** preserved throughout so every answer can cite exact sources